# 00 — Data Audit

Verify the integrity, completeness, and coordinate system of all raw input files before any processing.

**Goal:** Catch problems early. Do not proceed to `01_spatial_alignment` until this notebook passes cleanly.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## 1. Hotspot Catalog

In [ ]:
from ingest.hotspot_catalog import load_hotspot_catalog

try:
    catalog = load_hotspot_catalog()
    print(f"✅ Loaded {len(catalog)} hotspot records")
    display(catalog.head(10))
except FileNotFoundError as e:
    print(f"❌ {e}")
    catalog = None

In [ ]:
if catalog is not None:
    print("=== Column dtypes ===")
    print(catalog.dtypes)
    print()
    print("=== Missing values ===")
    print(catalog.isnull().sum())
    print()
    print("=== Coordinate ranges ===")
    print(f"Longitude: {catalog['longitude'].min():.2f} to {catalog['longitude'].max():.2f}")
    print(f"Latitude:  {catalog['latitude'].min():.2f} to {catalog['latitude'].max():.2f}")

In [ ]:
if catalog is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    catalog['longitude'].hist(bins=36, ax=axes[0], color='orange')
    axes[0].set_title('Longitude Distribution')
    axes[0].set_xlabel('Longitude (°)')
    
    catalog['latitude'].hist(bins=18, ax=axes[1], color='coral')
    axes[1].set_title('Latitude Distribution')
    axes[1].set_xlabel('Latitude (°)')
    
    if 'temperature' in catalog.columns:
        catalog['temperature'].dropna().hist(bins=20, ax=axes[2], color='red')
        axes[2].set_title('Temperature Distribution (K)')
    else:
        axes[2].text(0.5, 0.5, 'No temperature data', ha='center', transform=axes[2].transAxes)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n⚠️  NOTE: Latitude distribution reflects observational coverage bias,")
    print(f"   not true hotspot distribution. Southern hemisphere may be undersampled.")

## 2. Tidal Heating Grid

In [ ]:
from ingest.tidal_heating import load_tidal_heating_csv, load_synthetic_tidal_proxy
from config import RAW_DIR, TIDAL_HEATING_FILENAME

tidal_path = RAW_DIR / TIDAL_HEATING_FILENAME
if tidal_path.exists():
    tidal_df = load_tidal_heating_csv()
    print(f"✅ Loaded real tidal heating data: {len(tidal_df)} points")
else:
    print("⚠️  Real tidal heating data not found. Using synthetic proxy for development.")
    print("   See data/external/SOURCES.md for download instructions.")
    tidal_df = load_synthetic_tidal_proxy()
    print(f"   Synthetic proxy: {len(tidal_df)} points")

display(tidal_df.describe())

## 3. Geology Map

In [ ]:
from config import RAW_DIR, GEOLOGY_MAP_FILENAME

geo_path = RAW_DIR / GEOLOGY_MAP_FILENAME
if geo_path.exists():
    from ingest.geology_map import load_geology_map
    gdf = load_geology_map()
    print(f"✅ Loaded geology map: {len(gdf)} polygons")
    print(f"   CRS: {gdf.crs}")
    if 'unit_name' in gdf.columns:
        print(f"   Unique units: {gdf['unit_name'].nunique()}")
        print(gdf['unit_name'].value_counts().head(10))
else:
    print("⚠️  Geology shapefile not found.")
    print("   See data/external/SOURCES.md for download instructions.")

## 4. Audit Summary

**Checklist before proceeding:**

- [ ] Hotspot catalog loaded with expected columns
- [ ] Longitude normalized to [-180, 180]
- [ ] No missing coordinates
- [ ] Tidal heating data available (or synthetic proxy documented)
- [ ] Geology shapefile available (or absence noted)
- [ ] Observational bias noted in latitude distribution

**Document any issues found below:**

In [ ]:
# Audit notes — fill in before marking complete
audit_notes = """
Date: 
Analyst: 
Issues found: 
Assumptions made: 
"""